In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl
import os

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14})

In [ ]:
def load1dFig5(i):
    return np.loadtxt(os.path.join('..', '20221123 HMIA 13-3 Kondo', 'data', str(i), 'data.tsv'))

def load2dFig5(i, num):
    tmp = np.loadtxt(os.path.join('..', '20221123 HMIA 13-3 Kondo', 'data', str(i), 'data.tsv'))
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
import skunk
#import cairosvg
import matplotlib.lines as mlines
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.optimize import curve_fit
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec


In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

# Figure 6

In [ ]:
from scipy.special import gamma as Gamma
# import matplotlib.pyplot as plt
from scipy.special import digamma, hyp2f1
from scipy.integrate import quad
from scipy.optimize import fsolve

#Following equations defined in PhysRevB.56.1848
def JsumComponent(t, R, Ec, T):
    #The infinite sum component of J(t) defined in above paper
    gamma = np.euler_gamma
    beta = 1.0/T
    x=beta*Ec/(2*R*np.pi**2)
    y=np.exp(-4*np.pi**2 * t/beta)
    return -(1/np.pi)*(2*gamma+digamma(-x)+digamma(x)+2*np.log(1-y)+(y/(1+x)) *hyp2f1(1,1+x,2+x,y) + (y/(1-x)) * hyp2f1(1, 1-x,2-x,y))

def J(t, R, Ec, T):
    #R=series resistance
    #assuming charge of electron, h = 1 (so Rk = e^2/h = 1)
    #kboltzmann = 1
    #Also assuming Ec = 1/2C (got from Ingold and Nazarov Chapter in Single Charge Tunneling Book, eq 66)
    wc = 2*Ec/R
    beta = 1.0/T

    return np.pi*R*((1-np.exp(-wc*abs(t)))*(np.tan(beta*wc/(4*np.pi))**(-1) -1j)- 4*np.pi*abs(t)/beta + JsumComponent(abs(t), R, Ec,T))

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EB(V, R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1), dtype=np.longdouble)
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EBzero(R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    V = 0
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1), dtype=np.longdouble)
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])

def Nazarov(tau, EB):
    return (1-tau)*EB

def KNtheory(tau0, T, Ec, Z):
    idc = np.linspace(-2e-8, 2e-8, 81)
    V = 1*idc[31:51]*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

# def KNtheory(tau0, T, Ec, Z):
#     idc = np.linspace(-1.5e-8, 1.5e-8, 151)
#     V = 1*idc[50:101]*25813/3
#     tauV = np.zeros((len(V),))
#     EB0 = EB(0, Z, Ec, T)
#     EBV = EB(V, Z, Ec, T)
#     C = (tau0/(1-tau0))*(1/(EB0+1))
#     tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
#     return tauV


def KNtheory2(tau0, T, Ec, Z):
    idc = np.linspace(-1.5e-8, 1.5e-8, 151)
    V = 1*idc[30:120]*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

def KNtheoryfull(tau0, V, T, Ec, Z):
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV


In [ ]:
fig5 = plt.figure(figsize=(12, 8),constrained_layout=True)
gs = fig5.add_gridspec(2, 2, width_ratios=(1,1))
# gs = GridSpec(3, 3, figure=fig1)
f5_ax1 = fig5.add_subplot(gs[:1, 0])
f5_ax2 = fig5.add_subplot(gs[:1, 1])
f5_ax3 = fig5.add_subplot(gs[1:2, 0])
f5_ax4 = fig5.add_subplot(gs[1:2, 1])

# inset_ax = fig3.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')

f5_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f5_ax1.transAxes)
f5_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f5_ax2.transAxes)
f5_ax1.set_axis_off()
f5_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f5_ax3.transAxes)
f5_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f5_ax4.transAxes)

##################
dat = load2dFig5(138, 81)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vs = dat['3']
vr = dat['7']
Rl = np.mean(vs-vt-vr)/500e-12
vsdc = dat['10'] - idc*Rl
# vsdc = dat['9']
# vtdc = dat['10']

G = 3*vt/(vt+vr)
tauitop = (1/G - 1)**-1
start=60
fin=100

# Ginf at high neg bias
# ax.plot(botqpc[0,start:fin], tauitop[13, start:fin], 'b--', label='$V_{dc} = %1.1f \mu V$' %(idc[13,0]*25813*1e6/3)
f5_ax4.plot(botqpc[0,start:fin], tauitop[40, start:fin], 'k', label='$V_{dc} = %1.1f \mu V$' %(idc[40,0]*25813*1e6/3))
f5_ax4.plot(botqpc[0,start:fin], tauitop[-6, start:fin], 'b', label='$V_{dc} = %1.1f \mu V$' %(idc[-6,0]*25813*1e6/3))

f5_ax4.annotate("", xytext=(-3.04, 0.6), xy=(-3.04, 0.9),
            arrowprops=dict(arrowstyle="->"))

f5_ax4.grid(ls='--', lw=0.4)
f5_ax4.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')
f5_ax4.set_xlabel('$V_g$')

###################

dat = load2dFig5(138, 81)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vs = dat['3']
vr = dat['7']
Rl = np.mean(vs-vt-vr)/500e-12
# vsdc = dat['10'] - vsdc*Rl
# vsdc = dat['9']
# vtdc = dat['10']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauV = np.zeros(tau.shape)
start = 30
end = 52

start2 = start
start3 = 5

# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# im = ax[0].pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
# cbar = fig.colorbar(im, ax=ax[0], extend='max')
# ax[0].set_xlabel('V$_g$ (V)')
begin=6
end=-4
# tauV = KNtheory(tau, Z, Ec, T, V)
popt = np.zeros((3,151))
pcov = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# V = np.linspace(-3*Ec, 3*Ec,3001)
j=0
paramBounds = ([1e-6, 70e-6, 0.99999*Z], [8e-6, 80e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[41,i]
    if i>=60 and i<100:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        f5_ax2.plot(1e6*(idc[begin:end,i] - idc[41,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        popt[:,j], pcovi = curve_fit(KNtheory, tau0, tau[31:51,i], p0=[5e-6, 77e-6, Z], bounds=paramBounds)
        pcov[:,j] = np.diag(pcovi)
        print(popt[:,j])
        # tauV[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[41,i])*25813/3, 4e-6, 77e-6, 1)
        tauV[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[41,i])*25813/3, popt[0,j], popt[1,j], popt[2,j])
        # tauinf[j] = tauV[end-1,i]
        f5_ax2.plot(1e6*(idc[begin:end,i] - 1*idc[41,i])*25813/3, tauV[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')


f5_ax2.axvline(1e6*(idc[31,0] - idc[41,0])*25813/3, ls='--', color='gray')
f5_ax2.axvline(1e6*(idc[50,0] - idc[41,0])*25813/3, ls='--', color='gray')
# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
f5_ax2.set_xlabel('V$_{dc}$  ($\mu$V)')
f5_ax2.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')
f5_ax2.grid(ls='--', lw=0.4)

start=60
fin=100
f5_ax4.plot(botqpc[0,start:fin], tauV[-6,start:fin], 'm--',  label='Eq. 9, $V_{dc} = %1.1f \mu V$'%(idc[-6,0]*25813*1e6/3))
f5_ax4.legend()

# ax[1].set_ylim(0, 1.1)
# fig.savefig('figures/DCBtopQPConeRk.jpeg', dpi=300)
# fig.savefig('figures/DCBfitsKNmodel_oldZ1.jpeg', dpi=600)

###################
dat = load2dFig5(133, 81)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vs = dat['3']
vr = dat['7']
Rl = np.mean(vs-vt-vr)/500e-12
vsdc = dat['10'] - idc*Rl
# vsdc = dat['9']
# vtdc = dat['10']
startt = 66 #66
endt = 86 #66
EcbyTarray = np.linspace(1, 40, 101)
popt2 = np.zeros((3,151))
pcov2 = np.zeros((3,151))
G = 3*vt/(vt+vr)
tau = (1/G - 1/2)**-1
# f5_ax2.axvline(1e6*(idc[31,0] - idc[41,0])*25813/3, ls='--', color='gray')
# f5_ax2.axvline(1e6*(idc[50,0] - idc[41,0])*25813/3, ls='--', color='gray')

tauV2 = np.zeros(tau.shape)

begin=6
end=-4

Z = 0.5
j=0
paramBounds = ([1e-6, 70e-6, 0.99999*Z], [8e-6, 80e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[41,i]
    if i>=60 and i<100:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        # f5_ax2.plot(1e6*(idc[begin:end,i] - idc[41,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        popt2[:,j], pcovi2 = curve_fit(KNtheory, tau0, tau[31:51,i], p0=[5e-6, 77e-6, Z], bounds=paramBounds)
        pcov2[:,j] = np.diag(pcovi2)
        print(popt2[:,j])
        # tauV[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[41,i])*25813/3, 4e-6, 77e-6, 1)
        tauV2[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[41,i])*25813/3, popt2[0,j], popt2[1,j], popt2[2,j])
        # tauinf[j] = tauV[end-1,i]
        # f5_ax2.plot(1e6*(idc[begin:end,i] - 1*idc[41,i])*25813/3, tauV2[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1

n = idc.shape[1]-60-50
color = plt.cm.Blues(np.linspace(0.8, 0.8,20))
# mpl.rcParams['axes.prop_cycle'] = cycler('color', color)
# fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax = inset_axes(f5_ax3,
                    width="30%", # width = 30% of parent_bbox
                    height="30%", # height : 1 inch
                    bbox_to_anchor=(0.55,0.15,1.35, 1.35), bbox_transform=f5_ax3.transAxes,
                    loc=3)
ax.set_prop_cycle(cycler('color', color))
for i in range(idc.shape[1]):
    if i>60 and i%2==0 and i<100:
        if i==74:
            ax.plot(1e6*(idc[:,i] - idc[41,i])*25813/3, tau[:,i], color='r', label='$R_{env} = h/2e^2$')
        else:
            continue
#             ax.plot(1e6*idc[:,i]*25813/3, tau[:,i],)

# print(tau.shape)
minarg = np.argmin(tau[5:-5, :], axis = 0)
maxarg = np.argmax(tau[5:-5, :], axis = 0)
deltaG = np.zeros(tau.shape[1])
deltaG2 = np.zeros(tau.shape[1])
xdcb = np.zeros(tau.shape[1])
taumax = np.zeros(tau.shape[1])


for i in range(len(deltaG)):
    taumax[i] = tau[5+maxarg[i],i]
    deltaG[i] = tau[5+minarg[i],i] - tauV2[-6, i]
    xdcb[i] = tau[5+minarg[i],i]


# ax2.plot(xdcb[startt:], deltaG[startt:]/tau[60, startt:], 'bo', label='$R_{env} = R_K/2$, top qpc')
f5_ax3.plot(xdcb[startt:100], deltaG[startt:100]/tauV2[-6,startt:100], 'o', color='r', label='$R_{env} = h/2e^2$')

popt1, pcov1 = curve_fit(Nazarov, xdcb[startt:endt], deltaG[startt:endt]/tau[60, startt:endt])
print(popt1)
EcbyTfit1 = np.interp(popt1[0], np.flip(EBzero(0.5, EcbyTarray, 1, 0)), np.flip(EcbyTarray))
f5_ax3.plot(xdcb[:], Nazarov(xdcb[:], EBzero(0.5, 20*4e-6, 4e-6, 0)), ls='-.', color='r')#, label=f'$E_c/kT = {13:.1f}$')


f5_ax3.set_xlim(0, )
f5_ax3.grid(ls='--', lw=0.4)
f5_ax3.set_ylim(-1, 0)

dat = load2dFig5(138, 81)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vs = dat['3']
vr = dat['7']
Rl = np.mean(vs-vt-vr)/500e-12
vsdc = dat['10'] - idc*Rl

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1

n = idc.shape[1]-60-50
color = plt.cm.Reds(np.linspace(0.8, 0.8,20))
ax.set_prop_cycle(cycler('color', color))
for i in range(idc.shape[1]):
    if i>60 and i%2==0 and i<100:
        if i==68:
            ax.plot(1e6*(idc[:,i] - idc[41,i])*25813/3, tau[:,i], color='k', ls='--', label='$R_{env} = h/e^2$')
        else:
            continue
#             ax.plot(1e6*idc[:,i]*25813/3, tau[:,i], ls='--')

ax.set_xlabel('V$_{dc}$  ($\mu$V)')
ax.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')
# ax.set_ylim(0, 1.1)
# ax.legend(loc = 'upper right')

ax.grid(ls='--', lw=0.4)
# fig.tight_layout()

minarg = np.argmin(tau[5:-5, :], axis = 0)
maxarg = np.argmax(tau[5:-5, :], axis = 0)
deltaG = np.zeros(tau.shape[1])
xdcb = np.zeros(tau.shape[1])
taumax = np.zeros(tau.shape[1])


for i in range(len(deltaG)):
    taumax[i] = tau[5+maxarg[i],i]
    deltaG[i] = tau[5+minarg[i],i] - taumax[i]
    deltaG2[i] = tau[5+minarg[i],i] - tauV[-6,i]
    xdcb[i] = tau[5+minarg[i],i]

# for i in range(len(deltaG)):
#     deltaG[i] = tau[5+minarg[i],i] - tau[64, i]
#     xdcb[i] = tau[5+minarg[i],i]
# # fig2, ax2 = plt.subplots()
# f5_ax3.plot(xdcb[startt:], deltaG[startt:]/taumax[startt:], 's', color='k', label='$R_{env} = h/e^2$')
f5_ax3.plot(xdcb[startt:100], deltaG2[startt:100]/tauV[-6,startt:100], 's', color='k', label='$R_{env} = h/e^2$')

# popt2, pcov2 = curve_fit(Nazarov, xdcb[startt:endt], deltaG[startt:endt]/tau[60, startt:endt])
# print(popt2)
# EcbyTfit2 = np.interp(popt2[0], np.flip(EBzero(1, EcbyTarray, 1, 0)), np.flip(EcbyTarray))
# popt3, pcov3 = curve_fit(EBzero, xdcb[startt:endt], deltaG[startt:endt]/tau[60, startt:endt])
f5_ax3.plot(xdcb[:], Nazarov(xdcb[:], EBzero(1, 20*4e-6, 4e-6, 0)), ls='--', color='k')#, label=f'$E_c/kT = {13}$')
# Ec = 65e-6
# T = 5e-6
# print( EB(0, 1, Ec, T))
# ax2.plot(xdcb[startt:], Nazarov(xdcb[startt:], EB(0, 1, Ec, T)), ls='--', color='k')
# ax2.plot(xdcb, EB(), ls='--', color='k', label='$R_{env} = R_K$, top qpc')
f5_ax3.set_xlim(0, )
f5_ax3.set_ylim(-1, 0)
f5_ax3.set_xlabel('$\\tilde{\u03C4}_1(V_{dc} = 0)$')
f5_ax3.set_ylabel('$(\\tilde{\u03C4}_1(V_{dc} = 0) - \u03C4_1)/\u03C4_1$')
f5_ax3.legend(loc = 'upper left')
################

skunk.connect(f5_ax1, 'sk2')

# svg = skunk.pltsvg(fig=fig2)
svg = skunk.insert(
    {  
        'sk2': 'DCB.svg'
            
    })

# fig2.set_constrained_layout(False)
# fig3.tight_layout()

# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)


# # we don't want the layout to change at this point.
#fig1.tight_layout()

skunk.display(svg)
# cairosvg.svg2pdf(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure5_Dec2025.pdf')
# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")